# Topic Modeling de respuestas abiertas de encuestas educativas
### BERTopic + embeddings de `bertin-project/bertin-roberta-base-spanish` (BERTIN)

Cuaderno pensado para ejecutarse en **Google Colab**.

Este notebook complementa la comparativa de modelos de sentimiento en español, aplicando ahora un enfoque de **topic modeling** (extracción de tópicos) sobre respuestas abiertas de encuestas del sector educativo, usando BERTopic con embeddings generados a partir de RoBERTa-BNE.


## 1. Configuraciones Globales

### 1.1 Instalación de dependencias

In [1]:
!pip install -q bertopic transformers sentence-transformers umap-learn hdbscan scikit-learn nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.7 MB/s eta 0:00:00


In [2]:
# Modelo spacy
!pip install -q spacy
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 1.2 Imports

In [1]:
# --- Core ---
import os
import re
import random
import warnings

import numpy as np
import pandas as pd

# --- Deep learning ---
import torch

# --- Topic modeling ---
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# --- Utilidades ---
import nltk
from nltk.corpus import stopwords

# Nota: el import específico del pipeline de Transformers para generar
# embeddings con RoBERTa-BNE se define en la Sección 2 (Pipeline),
# una vez decidido el enfoque exacto (AutoModel/AutoTokenizer vs. pipeline).

warnings.filterwarnings("ignore")

### 1.3 Variables y objetos globales

In [2]:
# Modelo base para generar embeddings (RoBERTa-BNE)
# PlanTL-GOB-ES/roberta-base-bne fue retirado del Hub (repo oficial sin archivos disponibles).
# Se usa BERTIN, otro RoBERTa-base en espanol activo y completo, como reemplazo.
MODEL_NAME = "bertin-project/bertin-roberta-base-spanish"

# Dispositivo: usa GPU si Colab la asignó en este runtime, si no CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo en uso: {DEVICE}")

# Tamaño de lote por defecto para generación de embeddings en batch
# (se usará y podrá ajustarse en la Sección 2 - Pipeline)
BATCH_SIZE = 16

# Semilla para reproducibilidad (UMAP y HDBSCAN, usados internamente por BERTopic,
# tienen componentes estocásticos)
RANDOM_SEED = 42


def set_seed(seed: int = RANDOM_SEED) -> None:
    """Fija la semilla en random, numpy y torch (CPU y GPU) para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)

Dispositivo en uso: cpu


In [3]:
# Stopwords en español, usadas más adelante por el CountVectorizer de BERTopic
# para limpiar la representación final de los tópicos (palabras clave por tópico)
nltk.download("stopwords", quiet=True)
STOPWORDS_ES = stopwords.words("spanish")

print(f"Stopwords en español cargadas: {len(STOPWORDS_ES)}")

Stopwords en español cargadas: 313


In [4]:
import spacy

# Modelo de español de spaCy, desactivando componentes que no usamos
# (parser de dependencias y reconocedor de entidades) para acelerar el procesamiento.
nlp = spacy.load("es_core_news_md", disable=["parser", "ner"])

print("Modelo de spaCy cargado:", nlp.meta["name"], nlp.meta["version"])

Modelo de spaCy cargado: core_news_md 3.8.0


## 2. Carga del corpus real

### 2.1 Selección y carga del archivo CSV

In [5]:
from google.colab import files

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Archivo cargado: {csv_filename}")

Saving autoevaluacion_acreditacion__acred_sugerencias_complementarias.csv to autoevaluacion_acreditacion__acred_sugerencias_complementarias.csv
Archivo cargado: autoevaluacion_acreditacion__acred_sugerencias_complementarias.csv


In [6]:
# Lectura del CSV, con fallback de encoding (frecuente en CSVs exportados desde Excel en español)
try:
    df = pd.read_csv(csv_filename, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_filename, encoding="latin-1")

print(f"Filas cargadas: {len(df)}")
df.head()

Filas cargadas: 4157


,Periodo,ID de respuesta,llave,FNACNAANAi181i.,FNACNAANAi181i._clean,FNACNAANAi181i._no_stopwords,FNACNAANAi181i._no_stopwords_no_adverbs
0,2024-1,125648221,Estudiantes-BOGOTÁ-1115-2024-1,Considero que la facultad a veces cuando inici...,considero que la facultad a veces cuando inici...,considero veces inicia semestre muchas cosas d...,considero veces inicia semestre muchas cosas d...
1,2024-1,126030698,Estudiantes-BOGOTÁ-1115-2024-1,Me gustaría poder conocer mAs sobre los progra...,me gustaria poder conocer mas sobre los progra...,gustaria poder conocer mas programas ofrece in...,gustaria poder conocer programas ofrece intern...
2,2024-1,123966220,Estudiantes-BOGOTÁ-1116-2024-1,Las salones que nos han asignado casi siempre ...,las salones que nos han asignado casi siempre ...,salones asignado casi siempre suelen ser peque...,salones asignado suelen ser pequeños cantidad ...
3,2024-1,124294337,Estudiantes-BOGOTÁ-1116-2024-1,Me parece permiten que sea mAs visible la divu...,me parece permiten que sea mas visible la divu...,parece permiten mas visible divulgacion acerca...,parece permiten visible divulgacion acerca int...
4,2024-1,124417900,Estudiantes-BOGOTÁ-1116-2024-1,"Considero que la forma en que, las que hacen p...",considero que la forma en que las que hacen pa...,considero forma hacen parte administracion pat...,considero forma hacen parte administracion pat...


### 2.2 Limpieza y extracción de la columna de respuestas

In [7]:
# Columna del CSV que contiene las respuestas abiertas de la encuesta
TEXT_COLUMN = "FNACNAANAi181i."

# Limpieza: descartar nulos y strings vacíos, normalizar a texto
df_clean = df[df[TEXT_COLUMN].notna()].copy()
df_clean[TEXT_COLUMN] = df_clean[TEXT_COLUMN].astype(str).str.strip()
df_clean = df_clean[df_clean[TEXT_COLUMN] != ""]
df_clean.reset_index(drop=True, inplace=True)

# Lista de documentos que alimentará la generación de embeddings y BERTopic
documents = df_clean[TEXT_COLUMN].tolist()

print(f"Filas totales en el CSV: {len(df)}")
print(f"Respuestas válidas tras limpieza: {len(documents)}")
print("\nEjemplos de respuestas:")
for ejemplo in documents[:3]:
    print(f"- {ejemplo[:150]}...")

Filas totales en el CSV: 4157
Respuestas válidas tras limpieza: 4157

Ejemplos de respuestas:
- Considero que la facultad a veces cuando inicia semestre o esta con muchas cosas dejan de lado a los estudiantes y no les ayudan con dudas, tal vez te...
- Me gustaría poder conocer mAs sobre los programas que ofrece la universidad de internacionalización, en ocasiones aparecen convocatorias pero la infor...
- Las salones que nos han asignado casi siempre suelen ser muy pequeños para la cantidad de estudiantes, en clases donde se necesita pc es muy díficil q...


## 3. Pipeline

### 3.1 Imports específicos del pipeline

In [8]:
from transformers import AutoTokenizer, AutoModel
from umap import UMAP
from hdbscan import HDBSCAN
from tqdm.auto import tqdm

### 3.2 Carga del modelo de embeddings (BERTIN)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()  # modo inferencia: desactiva dropout

# Longitud máxima de tokens por respuesta (las respuestas de encuesta son más largas que un tuit)
MAX_LENGTH = 256

print(f"Modelo '{MODEL_NAME}' cargado en: {DEVICE}")

config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/851k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/509k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: bertin-project/bertin-roberta-base-spanish
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo 'bertin-project/bertin-roberta-base-spanish' cargado en: cpu


### 3.3 Funciones de generación de embeddings

`mean_pooling` combina los embeddings de todos los tokens de un comentario (ponderados por `attention_mask`, para ignorar el padding) en un único vector por comentario, ya que BERTIN no genera nativamente un embedding de oración como sí lo hacen los modelos de `sentence-transformers`.

In [10]:
def mean_pooling(model_output, attention_mask):
    """Combina los embeddings de tokens (last_hidden_state) en un único vector por texto,
    ponderando por attention_mask para ignorar los tokens de padding."""
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

In [11]:
def generate_embeddings(texts, batch_size=BATCH_SIZE, device=DEVICE, max_length=MAX_LENGTH, show_progress=True):
    """
    Genera embeddings de oración para una lista de textos, procesando en lotes (batch)
    para optimizar el uso de GPU.

    Parámetros:
        texts (list[str]): textos a codificar.
        batch_size (int): tamaño de lote (optimización de uso de GPU).
        device (str): "cuda" o "cpu".
        max_length (int): longitud máxima de tokens por texto (truncamiento).
        show_progress (bool): mostrar barra de progreso.

    Retorna:
        np.ndarray de forma (len(texts), hidden_size)
    """
    all_embeddings = []
    batches = range(0, len(texts), batch_size)
    if show_progress:
        n_batches = (len(texts) + batch_size - 1) // batch_size
        batches = tqdm(batches, total=n_batches, desc="Generando embeddings")

    model.to(device)

    for i in batches:
        batch_texts = texts[i:i + batch_size]
        encoded_input = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            model_output = model(**encoded_input)

        batch_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
        all_embeddings.append(batch_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


def embed_single(text):
    """Genera el embedding de un único texto (wrapper de generate_embeddings)."""
    return generate_embeddings([text], show_progress=False)[0]

### 3.4 Generación de embeddings del corpus

In [12]:
embeddings_corpus = generate_embeddings(documents, batch_size=BATCH_SIZE, device=DEVICE, max_length=MAX_LENGTH)
print(f"Embeddings generados: {embeddings_corpus.shape}")

Generando embeddings:   0%|          | 0/260 [00:00<?, ?it/s]

Embeddings generados: (4157, 768)


### 3.5 Definición y entrenamiento de BERTopic

`min_cluster_size=50` como punto de partida (≈1,2% del corpus de 4.200 respuestas), priorizando tópicos amplios y accionables; a revisar según los resultados de `get_topic_info()`.

In [13]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_SEED
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer()

topic_model = BERTopic(
    embedding_model=None,  # usamos embeddings precalculados con RoBERTa-BNE, no un modelo interno de BERTopic
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents, embeddings=embeddings_corpus)

2026-09-18 18:59:38,298 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-18 19:00:31,767 - BERTopic - Dimensionality - Completed ✓
2026-09-18 19:00:31,769 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-18 19:00:32,625 - BERTopic - Cluster - Completed ✓
2026-09-18 19:00:32,634 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-18 19:00:32,951 - BERTopic - Representation - Completed ✓


### 3.6 Resultados del entrenamiento

In [14]:
topic_info = topic_model.get_topic_info()

n_topicos = len(topic_info) - (1 if -1 in topic_info["Topic"].values else 0)
n_ruido = topic_info.loc[topic_info["Topic"] == -1, "Count"].values[0] if -1 in topic_info["Topic"].values else 0

print(f"Tópicos encontrados (excluyendo ruido): {n_topicos}")
print(f"Documentos clasificados como ruido (-1): {n_ruido} de {len(documents)} ({n_ruido / len(documents):.1%})")

topic_info

Tópicos encontrados (excluyendo ruido): 20
Documentos clasificados como ruido (-1): 608 de 4157 (14.6%)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,608,-1_con_destaco_de_el,"[con, destaco, de, el, la, trabajo, procesos, ...",[Me siento a gusto con los recursos físico y t...
1,0,1232,0_que_de_la_el,"[que, de, la, el, en, es, no, los, con, se]",[Disponer formas mAs efectivas de representaci...
2,1,459,1_encuesta_coordino_consolidados_sesgo,"[encuesta, coordino, consolidados, sesgo, cont...",[Yo no coordino ningún programa académico por ...
3,2,192,2_abiertas_vacantes_internas_convocatorias,"[abiertas, vacantes, internas, convocatorias, ...",[Las convocatorias internas sobre vacantes deb...
4,3,175,3_participar_pero_bienestar_en,"[participar, pero, bienestar, en, no, interesa...","[Agradezco la oportunidad, sería interesante y..."
5,4,151,4_de_la_los_ms,"[de, la, los, ms, formas, datos, investigacin,...",[Disponer formas más efectivas de representaci...
6,5,142,5_terrazas_abran_las_areas,"[terrazas, abran, las, areas, participan, juic...",[Para enriquecer este ejercicio de autoevaluac...
7,6,141,6_funciones_levante_manual_talento,"[funciones, levante, manual, talento, claras, ...",[En pro de que las funciones estén claras segú...
8,7,119,7_gracias_acuerdo_de_,"[gracias, acuerdo, de, , , , , , , ]","[gracias, gracias, gracias]"
9,8,119,8_neurodivergente_evale_oportuno_brindar,"[neurodivergente, evale, oportuno, brindar, pa...",[Me parecería oportuno que se evalúe la posibi...


### 3.6.1 Recálculo de palabras clave con texto limpio (sin tildes, stopwords, adverbios)

In [15]:
def extraer_sustantivos(textos, batch_size=64, n_process=1):
    """
    Filtra cada texto dejando solo sustantivos comunes y propios (NOUN, PROPN),
    lematizados y en minúscula. Procesa en lotes con nlp.pipe para eficiencia.
    Pensado para aplicarse sobre texto ya limpio (sin stopwords, sin echowords, sin adverbios).
    """
    resultados = []
    for doc in nlp.pipe(textos, batch_size=batch_size, n_process=n_process):
        lemas = [token.lemma_.lower() for token in doc if token.pos_ in ("NOUN", "PROPN")]
        resultados.append(" ".join(lemas))
    return resultados

In [16]:
# 1. Guardamos los Representative_Docs originales (calculados con texto original en el fit_transform),
#    porque update_topics() los va a sobrescribir con los documentos que le pasemos.
representative_docs_originales = topic_model.representative_docs_

# 2. Partimos de la columna ya limpia (sin tildes, stopwords, adverbios, echowords)
documentos_limpios = df_clean["FNACNAANAi181i._no_stopwords_no_adverbs"].fillna("").astype(str).tolist()

# 3. Sobre ESA columna limpia, dejamos solo sustantivos (comunes y propios), lematizados
documentos_sustantivos = extraer_sustantivos(documentos_limpios)

# 4. Vectorizador simple: ya no hace falta stop_words, el filtro de sustantivos ya dejó solo eso
vectorizer_model_sustantivos = CountVectorizer()

# 5. Recalculamos las palabras clave (c-TF-IDF) usando el texto filtrado a sustantivos,
#    sin tocar la asignación de tópicos ya hecha
topic_model.update_topics(documentos_sustantivos, vectorizer_model=vectorizer_model_sustantivos)

# 6. Restauramos los Representative_Docs originales (en texto original), que update_topics() acaba de sobrescribir
topic_model.representative_docs_ = representative_docs_originales

# 7. Verificación: palabras clave actualizadas
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,608,-1_proceso_calidez_receptividad_eficiencia,"[proceso, calidez, receptividad, eficiencia, d...",[Me siento a gusto con los recursos físico y t...
1,0,1232,0_programa_tema_estudiante_materia,"[programa, tema, estudiante, materia, experien...",[Disponer formas mAs efectivas de representaci...
2,1,459,1_sesgo_resultado_mayoria_pregunta,"[sesgo, resultado, mayoria, pregunta, respuest...",[Yo no coordino ningún programa académico por ...
3,2,192,2_vacante_convocatoria_neurodivergente_acompaa...,"[vacante, convocatoria, neurodivergente, acomp...",[Las convocatorias internas sobre vacantes deb...
4,3,175,3_bienestar_campos_generalizo_remoto,"[bienestar, campos, generalizo, remoto, consid...","[Agradezco la oportunidad, sería interesante y..."
5,4,151,4_forma_clase_vez_complicacion,"[forma, clase, vez, complicacion, formalismo, ...",[Disponer formas más efectivas de representaci...
6,5,142,5_terraza_juicio_gerencia_area,"[terraza, juicio, gerencia, area, proceso, mes...",[Para enriquecer este ejercicio de autoevaluac...
7,6,141,6_funcin_levante_manual_talento,"[funcin, levante, manual, talento, segun, depa...",[En pro de que las funciones estén claras segú...
8,7,119,7_gracia_acuerdo__,"[gracia, acuerdo, , , , , , , , ]","[gracias, gracias, gracias]"
9,8,119,8_neurodivergente_acompaamiento_posibilidad_ad...,"[neurodivergente, acompaamiento, posibilidad, ...",[Me parecería oportuno que se evalúe la posibi...


### 3.7 Guardado del modelo y de los embeddings en Google Drive

Persiste el modelo `BERTopic` entrenado y la matriz de `embeddings_corpus` en Google Drive, para no tener que repetir el cómputo (embeddings + entrenamiento) en cada sesión de Colab.

In [17]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Carpeta destino en Google Drive (ajustar la ruta si se prefiere otra ubicación)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/--topic modeling calidad"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"Carpeta de salida: {DRIVE_OUTPUT_DIR}")

Carpeta de salida: /content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/--topic modeling calidad


In [19]:
# Guardado del modelo BERTopic (serialización "safetensors": recomendada por BERTopic, portable entre versiones)
# save_embedding_model=False porque el modelo se instanció con embedding_model=None
# (usamos embeddings precalculados con BERTIN, no un modelo interno de BERTopic)
topic_model_path = os.path.join(DRIVE_OUTPUT_DIR, "bertopic_model")
topic_model.save(
    topic_model_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)
print(f"Modelo BERTopic guardado en: {topic_model_path}")

2026-09-18 19:05:04,283 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Modelo BERTopic guardado en: /content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/--topic modeling calidad/bertopic_model


In [20]:
# Guardado de los embeddings del corpus, para reutilizarlos sin recalcularlos
embeddings_path = os.path.join(DRIVE_OUTPUT_DIR, "embeddings_corpus.npy")
np.save(embeddings_path, embeddings_corpus)
print(f"Embeddings guardados en: {embeddings_path} (forma: {embeddings_corpus.shape})")

Embeddings guardados en: /content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/--topic modeling calidad/embeddings_corpus.npy (forma: (4157, 768))


## 4. Prueba a 1 comentario

### 4.1 Comentario de ejemplo

Pega manualmente un comentario de ejemplo (idealmente uno no visto durante el entrenamiento) en la variable `comentario_ejemplo` de la celda siguiente.

In [21]:
comentario_ejemplo = "Disponer formas más efectivas de representación de los profesores en procesos decisionales de carácter académico que los afectan directamente (en la selección de nuevos docentes, en el desempeño de la labor docente y de investigación, en la relación con los estudiantes, en el tratamiento económico). Limitar la ocupación de la sede antigua con pabellones de ferias, eventos y uso de parlantes durante horas de clase. Más atención a los salones asignados para clases de pregrado (a veces no aptos, oscuros, obsoletos). Evitar la progresiva complicación, mitigar el exceso de formalismo y burocracia en los procedimientos administrativos (particularmente de cara a los requerimientos y estándares gubernamentales, cuyo cumplimiento ha significado una esclerotización e involución de la actividad de las unidades académicas de la Facultad, además de la acogida de criterios y principios no congruentes con el espíritu y la misión fundacional del Externado). Unificar bases de datos para no tener que realizar más de una vez recolección de los mismos datos para distintos propósitos y destinatarios. Mejorar la visibilización de los proyectos de investigación de la Universidad utilizando distintas formas de comunicación o plataformas (para público especializado y no)."  # <-- Pega aquí el comentario de ejemplo

assert comentario_ejemplo.strip() != "", "Completa la variable 'comentario_ejemplo' antes de continuar."
print(comentario_ejemplo)

Disponer formas más efectivas de representación de los profesores en procesos decisionales de carácter académico que los afectan directamente (en la selección de nuevos docentes, en el desempeño de la labor docente y de investigación, en la relación con los estudiantes, en el tratamiento económico). Limitar la ocupación de la sede antigua con pabellones de ferias, eventos y uso de parlantes durante horas de clase. Más atención a los salones asignados para clases de pregrado (a veces no aptos, oscuros, obsoletos). Evitar la progresiva complicación, mitigar el exceso de formalismo y burocracia en los procedimientos administrativos (particularmente de cara a los requerimientos y estándares gubernamentales, cuyo cumplimiento ha significado una esclerotización e involución de la actividad de las unidades académicas de la Facultad, además de la acogida de criterios y principios no congruentes con el espíritu y la misión fundacional del Externado). Unificar bases de datos para no tener que re

### 4.2 Generación del embedding del comentario

In [22]:
embedding_ejemplo = embed_single(comentario_ejemplo)
print(f"Embedding generado, forma: {embedding_ejemplo.shape}")

Embedding generado, forma: (768,)


### 4.3 Asignación de tópico con el modelo entrenado

In [23]:
topic_asignado, prob_asignada = topic_model.transform(
    [comentario_ejemplo],
    embeddings=embedding_ejemplo.reshape(1, -1)
)

topico = topic_asignado[0]

# Como el modelo se entrenó con calculate_probabilities=True, prob_asignada[0]
# es un array con la probabilidad de pertenecer a CADA tópico, no un único valor.
# Tomamos la probabilidad más alta de ese array (la del tópico asignado).
probabilidad = float(np.max(prob_asignada[0]))

print(f"Comentario: {comentario_ejemplo}\n")

if topico == -1:
    print("El comentario fue clasificado como RUIDO (-1): no encajó en ningún tópico existente.")
else:
    print(f"Tópico asignado: {topico}")
    print(f"Probabilidad: {probabilidad:.2%}\n")
    print("Palabras clave del tópico:")
    for palabra, peso in topic_model.get_topic(topico):
        print(f"  - {palabra} ({peso:.4f})")

2026-09-18 19:05:58,695 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-09-18 19:06:16,479 - BERTopic - Dimensionality - Completed ✓
2026-09-18 19:06:16,480 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-09-18 19:06:16,482 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-09-18 19:06:16,485 - BERTopic - Probabilities - Completed ✓
2026-09-18 19:06:16,486 - BERTopic - Cluster - Completed ✓


Comentario: Disponer formas más efectivas de representación de los profesores en procesos decisionales de carácter académico que los afectan directamente (en la selección de nuevos docentes, en el desempeño de la labor docente y de investigación, en la relación con los estudiantes, en el tratamiento económico). Limitar la ocupación de la sede antigua con pabellones de ferias, eventos y uso de parlantes durante horas de clase. Más atención a los salones asignados para clases de pregrado (a veces no aptos, oscuros, obsoletos). Evitar la progresiva complicación, mitigar el exceso de formalismo y burocracia en los procedimientos administrativos (particularmente de cara a los requerimientos y estándares gubernamentales, cuyo cumplimiento ha significado una esclerotización e involución de la actividad de las unidades académicas de la Facultad, además de la acogida de criterios y principios no congruentes con el espíritu y la misión fundacional del Externado). Unificar bases de datos para no 

## Cómo leer los pesos (c-TF-IDF)

BERTopic no usa TF-IDF tradicional, sino una variante llamada **c-TF-IDF** ("class-based TF-IDF"): trata a **todos los comentarios de un mismo tópico como si fueran un solo documento gigante**, y calcula qué tan frecuente es cada palabra *dentro de ese tópico* en comparación con su frecuencia en *todos los demás tópicos combinados*.

Eso significa:

- **Una palabra con peso alto** aparece mucho en los comentarios de ese tópico específico, y poco en los de los demás tópicos — es decir, es una palabra que **distingue** a ese tópico del resto, no solo una palabra frecuente en general (por eso "más" o "de" no aparecen, aunque sean comunes: los filtra el `CountVectorizer` con `STOPWORDS_ES`, y además no serían distintivas de un tópico en particular).
- **El orden importa más que el número absoluto**: BERTopic ya te las entrega ordenadas de mayor a menor relevancia para ese tópico. La primera palabra es la más representativa, y así sucesivamente.
- **El valor numérico (0.0510, 0.0506, etc.) no es una probabilidad ni un porcentaje** — es un puntaje relativo de importancia. No tiene una interpretación intuitiva como "51% de las veces" ni nada similar; solo sirve para *rankear* dentro de esa lista.
- **No se pueden comparar los pesos entre tópicos distintos** (por ejemplo, el 0.0510 del Tópico 4 contra el peso más alto del Tópico 7) — la escala depende del tamaño y vocabulario de cada tópico. Solo tiene sentido comparar posiciones *dentro* de la lista de un mismo tópico.

En tu caso, el Tópico 4 parece girar en torno a: procesos de decisión/formalismo institucional, estándares, investigación y manejo de datos — coherente con el comentario que probaste.

## Algo que noté (y que sí conviene revisar)

Las palabras clave salieron **sin tildes ni caracteres especiales**: `ms` (en vez de "más"), `investigacin` (en vez de "investigación"), `carcter` ("carácter"), `estndares` ("estándares"), `espritu` ("espíritu"). No es solo que falte el acento visual (como pasaría con á→a) — la letra desapareció por completo de la palabra.

Esto es interesante porque **tu comentario de ejemplo sí tiene tildes correctas** (lo ves en el "Comentario:" impreso arriba). Eso apunta a que el problema no está en BERTopic ni en el `CountVectorizer`, sino en **el corpus del CSV que cargaste en la Sección 2** — probablemente ya venía con esa pérdida de caracteres especiales desde el archivo original (un problema común de encoding en exportaciones de encuestas).

¿Quieres que agreguemos una celda de verificación en la Sección 2 para confirmar si el problema viene del CSV original (y así decidir si hay que ajustar el encoding de lectura)?

## 5. Exportación de resultados a Excel

### 5.1 Instalación de dependencia

In [24]:
!pip install -q openpyxl

### 5.2 Reconstrucción de las filas válidas sobre el CSV original

In [25]:
# Reproducimos exactamente el mismo filtro de la Sección 2.2, mismo orden,
# para poder ubicar cada valor de 'topics' en su fila correspondiente del CSV original.
mask_validas = df[TEXT_COLUMN].notna() & (df[TEXT_COLUMN].astype(str).str.strip() != "")

assert mask_validas.sum() == len(topics), (
    f"Los conteos no coinciden: {mask_validas.sum()} filas válidas en 'df' vs {len(topics)} tópicos calculados. "
    "Verifica que 'df' y 'topics' correspondan al mismo CSV cargado en esta sesión."
)

print(f"Filas válidas (con tópico asignado): {mask_validas.sum()}")
print(f"Filas sin respuesta (descartadas en la limpieza): {(~mask_validas).sum()}")

Filas válidas (con tópico asignado): 4157
Filas sin respuesta (descartadas en la limpieza): 0


### 5.3 Formato de topic y Representation a partir de get_topic_info()

In [26]:
topic_info = topic_model.get_topic_info()

def formatear_topic(row):
    # 'Name' ya trae el número al inicio (ej: "4_forma_representacion_profesor_proceso");
    # lo separamos para armar "4 - forma_representacion_profesor_proceso"
    nombre_sin_numero = row["Name"].split("_", 1)[1] if "_" in row["Name"] else row["Name"]
    return f'{row["Topic"]} - {nombre_sin_numero}'

def formatear_representation(row):
    return ", ".join(row["Representation"])

topic_info["topic_formateado"] = topic_info.apply(formatear_topic, axis=1)
topic_info["representation_formateada"] = topic_info.apply(formatear_representation, axis=1)

mapa_topic = dict(zip(topic_info["Topic"], topic_info["topic_formateado"]))
mapa_representation = dict(zip(topic_info["Topic"], topic_info["representation_formateada"]))

### 5.4 Construcción del DataFrame final y guardado en Excel

In [27]:
df_resultado = df.copy()

# Valor por defecto para filas sin respuesta
df_resultado["topic"] = "Sin respuesta"
df_resultado["Representation"] = "Sin respuesta"

# Asignación para las filas válidas, usando los tópicos ya calculados en el fit_transform
df_resultado.loc[mask_validas, "topic"] = [mapa_topic[t] for t in topics]
df_resultado.loc[mask_validas, "Representation"] = [mapa_representation[t] for t in topics]

# Guardado en Google Drive, en la misma carpeta usada para el modelo y los embeddings
excel_filename = "respuestas_encuesta_con_topicos.xlsx"
excel_path = os.path.join(DRIVE_OUTPUT_DIR, excel_filename)

df_resultado.to_excel(excel_path, index=False)

print(f"Archivo guardado en: {excel_path}\n")
print(df_resultado["topic"].value_counts())

Archivo guardado en: /content/drive/MyDrive/Maestria_MINE/SEMESTRE 4/3.OpcionGrado--Wilmer/--topic modeling calidad/respuestas_encuesta_con_topicos.xlsx

topic
0 - programa_tema_estudiante_materia                            1232
-1 - proceso_calidez_receptividad_eficiencia                     608
1 - sesgo_resultado_mayoria_pregunta                             459
2 - vacante_convocatoria_neurodivergente_acompaamiento           192
3 - bienestar_campos_generalizo_remoto                           175
4 - forma_clase_vez_complicacion                                 151
5 - terraza_juicio_gerencia_area                                 142
6 - funcin_levante_manual_talento                                141
7 - gracia_acuerdo__                                             119
8 - neurodivergente_acompaamiento_posibilidad_administrativo     119
9 - programa_calidez_receptividad_eficiencia                     114
10 - bienestar_campos_remoto_generalizo                          103
11 - cargo_f